# 01 · Exploración y consolidación de los datos

Notebook de apoyo a la sección 1 del informe de la Segunda entrega. Toda la lógica vive en `src/desercion`; aquí solo se inspecciona. Requiere haber ejecutado `python -m desercion.cli prepare`.

In [1]:
import pandas as pd
from desercion import data
from desercion.features import feature_lists, EXCLUDED
pd.set_option('display.width', 160)
df = data.load_clean()
split = data.get_split(df)
df.shape

(77517, 52)

## Objetivo: deserción en el primer año por cohorte y momento

In [2]:
pd.concat([df.groupby('generation')[data.TARGET].agg(['size', 'mean']),
           pd.crosstab(df.generation, df['dropout.semester'])], axis=1)

,size,mean,0,1,2,3,4
generation,,,,,,,
AD14,10143,0.100858,9120,216,307,103,397
AD15,10041,0.092023,9117,456,0,99,369
AD16,10742,0.088159,9795,215,267,96,369
AD17,10788,0.087134,9848,245,251,107,337
AD18,11296,0.075336,10445,140,250,139,322
AD19,12199,0.094270,11049,93,572,82,403
AD20,12308,0.079461,11330,163,386,66,363


## Evidencia de fuga en las actividades co-curriculares
El valor *No information* concentra desertores: el registro falta *porque* el estudiante se fue.

In [3]:
pd.crosstab(df['total.life.activities'], df[data.TARGET], normalize='index').round(3)

desercion,0,1
total.life.activities,,
0,0.878,0.122
1,0.913,0.087
2,0.925,0.075
3,0.941,0.059
4,0.941,0.059
5,0.962,0.038
Does not apply,0.908,0.092
No information,0.255,0.745


## Desempeño del primer periodo según el momento de la deserción (AD19–AD20)
Quienes desertan *durante* el primer semestre (código 1) tienen promedios muy bajos porque el abandono ocurre dentro del periodo; por eso se excluyen del escenario B.

In [4]:
b = df[df.generation.isin(['AD19', 'AD20'])]
b.groupby('dropout.semester')[['average.first.period', 'failed.subject.first.period',
                               'dropped.subject.first.period']].mean().round(2)

,average.first.period,failed.subject.first.period,dropped.subject.first.period
dropout.semester,,,
0,89.67,0.03,0.02
1,54.89,0.02,0.05
2,77.27,0.24,0.07
3,82.70,0.15,0.05
4,85.23,0.11,0.05


## Tamaños por escenario y partición

In [5]:
pd.read_csv('../reports/tables/datos_resumen.csv')

,escenario,parte,registros,estudiantes,desertores,tasa_desercion
0,A,dev,62014,61950,5451,0.087900
1,A,test,15503,15487,1362,0.087854
2,B,dev,19408,19403,1505,0.077545
3,B,test,4843,4842,367,0.075779


## Variables excluidas y motivo

In [6]:
pd.Series(EXCLUDED, name='motivo').to_frame()

,motivo
retention,Es el objetivo (se recodifica como desercion =...
dropout.semester,Derivada del objetivo: solo es > 0 si el estud...
student.id,Identificador; se usa solo para agrupar la par...
generation,Identifica la cohorte; una cohorte nueva no ex...
level,Constante tras filtrar pregrado.
id.school.origin,6.534 niveles; su información se resume en sch...
father.education.complete,Redundante con father.education.summary.
mother.education.complete,Redundante con mother.education.summary.
father.exatec,Redundante con parents.exatec.
mother.exatec,Redundante con parents.exatec.
